# Olist - Relacionamento entre as tabelas

Modelo relacional em estrela ao redor de `orders`.

```mermaid
erDiagram
    CUSTOMERS ||--o{ ORDERS : "faz"
    ORDERS ||--o{ ORDER_ITEMS : "contem"
    ORDERS ||--o{ ORDER_PAYMENTS : "paga"
    ORDERS ||--o{ ORDER_REVIEWS : "recebe"
    PRODUCTS ||--o{ ORDER_ITEMS : "vendido em"
    SELLERS ||--o{ ORDER_ITEMS : "vende"
    PRODUCTS }o--|| PRODUCT_CATEGORY_TRANSLATION : "traduz"
    CUSTOMERS }o--o{ GEOLOCATION : "zip code"
    SELLERS }o--o{ GEOLOCATION : "zip code"

    CUSTOMERS {
        string customer_id PK
        string customer_unique_id
        int customer_zip_code_prefix FK
    }
    ORDERS {
        string order_id PK
        string customer_id FK
        string order_status
    }
    ORDER_ITEMS {
        string order_id FK
        int order_item_id
        string product_id FK
        string seller_id FK
    }
    ORDER_PAYMENTS {
        string order_id FK
        int payment_sequential
    }
    ORDER_REVIEWS {
        string review_id PK
        string order_id FK
    }
    PRODUCTS {
        string product_id PK
        string product_category_name FK
    }
    SELLERS {
        string seller_id PK
        int seller_zip_code_prefix FK
    }
    GEOLOCATION {
        int geolocation_zip_code_prefix
    }
    PRODUCT_CATEGORY_TRANSLATION {
        string product_category_name PK
        string product_category_name_english
    }
```

Observação: `customer_id` identifica um *pedido* do cliente, não a pessoa — quem identifica a pessoa é `customer_unique_id` (um cliente pode ter vários `customer_id`, um por pedido).

In [ ]:
import pandas as pd

from module_olist.config import RAW_DATA_DIR

orders = pd.read_csv(RAW_DATA_DIR / "olist_orders_dataset.csv")
customers = pd.read_csv(RAW_DATA_DIR / "olist_customers_dataset.csv")
items = pd.read_csv(RAW_DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(RAW_DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DATA_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW_DATA_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DATA_DIR / "olist_sellers_dataset.csv")
geo = pd.read_csv(RAW_DATA_DIR / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(RAW_DATA_DIR / "product_category_name_translation.csv")

## Integridade referencial

Para cada relação, conta quantas chaves do lado "filho" não existem no lado "pai" (órfãos) e quantas linhas do filho repetem a chave (indicando 1:N).

In [ ]:
def check_relationship(name: str, child: pd.DataFrame, child_col: str, parent: pd.DataFrame, parent_col: str) -> dict:
    orphans = (~child[child_col].isin(parent[parent_col])).sum()
    dup_child_key = child[child_col].duplicated().sum()
    return {
        "relacao": name,
        "orfaos": orphans,
        "linhas_filho_com_chave_repetida": dup_child_key,
        "total_linhas_filho": len(child),
    }


results = [
    check_relationship("orders -> customers", orders, "customer_id", customers, "customer_id"),
    check_relationship("items -> orders", items, "order_id", orders, "order_id"),
    check_relationship("items -> products", items, "product_id", products, "product_id"),
    check_relationship("items -> sellers", items, "seller_id", sellers, "seller_id"),
    check_relationship("payments -> orders", payments, "order_id", orders, "order_id"),
    check_relationship("reviews -> orders", reviews, "order_id", orders, "order_id"),
    check_relationship("customers -> geolocation (zip)", customers, "customer_zip_code_prefix", geo, "geolocation_zip_code_prefix"),
    check_relationship("sellers -> geolocation (zip)", sellers, "seller_zip_code_prefix", geo, "geolocation_zip_code_prefix"),
    check_relationship(
        "products -> category_translation",
        products.dropna(subset=["product_category_name"]),
        "product_category_name",
        category_translation,
        "product_category_name",
    ),
]

pd.DataFrame(results)

## Cardinalidade

In [ ]:
print("orders.order_id e unico:", orders["order_id"].is_unique)
print("customers.customer_id e unico:", customers["customer_id"].is_unique)
print("itens por pedido (media):", items.groupby("order_id").size().mean().round(2))
print("pagamentos por pedido (media):", payments.groupby("order_id").size().mean().round(2))
print("reviews por pedido (max):", reviews.groupby("order_id").size().max())
print(
    "clientes (customer_unique_id) com mais de um customer_id:",
    customers.groupby("customer_unique_id")["customer_id"].nunique().gt(1).sum(),
)